# LoRA for Gemma 4 in Colab

This notebook trains a 4-bit QLoRA adapter for Gemma 4 26B-A4B or Gemma 4 31B with `train/lora.py`. Run the cells in order. Checkpoints and the adapter go to Google Drive, so after a disconnect you run every cell again and training continues from the last checkpoint.

Choose the model. `26B-A4B` is the mixture-of-experts model served on the Thor; `31B` is the dense model.

In [ ]:
MODEL = "google/gemma-4-26B-A4B-it"  # or "google/gemma-4-31B-it"
RUN_NAME = MODEL.split("/")[1]

Check the GPU. An A100 or H100 with 80 GB trains 4,096-token examples; a 40 GB card needs 2,048. Smaller cards run out of memory.

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{gpu_name}, {gpu_gb:.0f} GB")

MAX_LENGTH = 4096 if gpu_gb > 70 else 2048
if gpu_gb < 39:
    print("WARNING: this GPU is too small; choose an A100 or H100 under Runtime > Change runtime type")

Mount Google Drive. The data, the checkpoints and the adapter live in `MyDrive/ttp`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/ttp"
RUN_DIR = f"{DRIVE_DIR}/runs/lora-{RUN_NAME}"
!mkdir -p {DRIVE_DIR}/runs

Clone the repository and install the pinned training packages. torch comes with Colab.

In [ ]:
!test -d ttp || git clone --depth 1 https://github.com/arpanpathak/thor-thunder-tigress-platform ttp
!pip install -q -r ttp/train/requirements.txt

Sign in to Hugging Face. Gemma models are gated: accept the licence on the model page first. A Colab secret named `HF_TOKEN` skips the prompt.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Put the training data on Drive as `MyDrive/ttp/train.jsonl`. When the file is not there yet, this cell asks for it and copies the upload to Drive. To train on several files, list them in `DATA`.

In [ ]:
import os
import shutil

from google.colab import files

DATA = [f"{DRIVE_DIR}/train.jsonl"]

if not os.path.exists(DATA[0]):
    uploaded = files.upload()
    shutil.copy(next(iter(uploaded)), DATA[0])

!wc -l {' '.join(DATA)}

Train. A checkpoint goes to Drive every 50 steps, and `--resume` starts from the newest one, so this cell also continues a run that a disconnect stopped.

In [ ]:
!python ttp/train/lora.py --model {MODEL} --precision 4bit \
    --data {' '.join(DATA)} --output-dir {RUN_DIR} \
    --max-length {MAX_LENGTH} --epochs 1 --save-steps 50 --resume

The adapter is in `RUN_DIR` on Drive. Copy that folder to the Thor and merge it there with `lora.py --merge` (see `train/README.md`).

In [ ]:
!ls -la {RUN_DIR}

Ask the trained model one tool-calling question. The answer should be a `<|tool_call>call:read_file{...}` that reads `src/main.rs`, not a guess about what the file prints.

In [ ]:
!python ttp/train/lora.py --generate --model {MODEL} --precision 4bit \
    --adapter {RUN_DIR} --new-tokens 120